## Cell 1 — Verify A100

Check that the runtime is using an A100 GPU before running any experiment. If the GPU name does not contain 'A100', switch the runtime in **Runtime → Change runtime type → Hardware accelerator → A100** and reconnect.

In [ ]:
# Run cells in order -- this is Cell 1 of 12
import torch

print('CUDA available : ' + str(torch.cuda.is_available()))
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    free_vram = torch.cuda.mem_get_info()[0] / 1e9
    print('GPU name       : ' + gpu_name)
    print(f'Free VRAM      : {free_vram:.1f} GB')
    if 'A100' not in gpu_name:
        print()
        print('WARNING: This runtime is NOT using an A100.')
        print('Go to Runtime > Change runtime type > Hardware accelerator > A100')
        print('Do not continue until you are on an A100 -- running on T4/V100 wastes CU budget.')
    else:
        print()
        print('A100 confirmed. Ready to proceed.')
else:
    print()
    print('WARNING: No GPU detected. Assign a GPU runtime before continuing.')

## Cell 2 — Install Dependencies

Install all required Python packages: timm, torchattacks, bitsandbytes, optimum, and pyyaml.

In [ ]:
# Run cells in order -- this is Cell 2 of 12
!pip install -q timm torchattacks bitsandbytes optimum pyyaml
print('Dependencies installed.')

## Cell 3 — Mount Google Drive

Mount Google Drive at /content/drive so that results can be backed up and restored across Colab sessions.

In [ ]:
# Run cells in order -- this is Cell 3 of 12
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## Cell 4 — Clone Repository

Clone the ADVC repository from GitHub. **Replace `YOUR_USERNAME` with your actual GitHub username before running this cell.**

In [ ]:
# Run cells in order -- this is Cell 4 of 12
# IMPORTANT: Replace Jmanav below with your actual GitHub username if different.
import subprocess, os, sys

GITHUB_USERNAME = 'Jmanav'  # <-- change this if your username is different
REPO_URL = 'https://github.com/' + GITHUB_USERNAME + '/ADVC.git'
CLONE_DIR = '/content/ADVC'

if os.path.exists(os.path.join(CLONE_DIR, '.git')):
    print('Repo already cloned -- pulling latest changes.')
    result = subprocess.run(['git', 'pull'], cwd=CLONE_DIR, capture_output=True, text=True)
    print(result.stdout or '(nothing to pull)')
    if result.returncode != 0:
        print('git pull error: ' + result.stderr)
        sys.exit(1)
else:
    print('Cloning ' + REPO_URL + ' ...')
    result = subprocess.run(
        ['git', 'clone', REPO_URL, CLONE_DIR],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print('git clone FAILED. Error output:')
        print(result.stderr)
        print()
        print('Common causes:')
        print('  1. Wrong username -- current value: ' + GITHUB_USERNAME)
        print('  2. Repo is private and requires a personal access token')
        print('  3. Network error -- try re-running the cell')
        raise RuntimeError('git clone failed -- see error above')
    print(result.stdout)

if not os.path.exists(CLONE_DIR):
    raise RuntimeError('Clone appeared to succeed but ' + CLONE_DIR + ' still does not exist.')

os.chdir(CLONE_DIR)
print('Working directory: ' + os.getcwd())

## Cell 5 — Create Directories

Create the results, checkpoint, and figures directories under /content/ADVC/results if they do not already exist.

In [ ]:
# Run cells in order -- this is Cell 5 of 12
import os

dirs = [
    '/content/ADVC/results',
    '/content/ADVC/results/checkpoints/at',
    '/content/ADVC/results/checkpoints/atkd',
    '/content/ADVC/results/figures',
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print('Ready: ' + d)

print()
print('All directories ready.')

## Cell 6 — Restore Previous Results from Drive

Copy any previously saved CSV result files from /content/drive/MyDrive/research back into the results directory, enabling resumption of an interrupted run.

In [ ]:
# Run cells in order -- this is Cell 6 of 12
import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/research'
RESULTS_DIR = '/content/ADVC/results'

files = [
    'phase1_results.csv',
    'phase2_at_results.csv',
    'phase2_atkd_results.csv',
    'phase3_results.csv',
]

restored = []
for f in files:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join(RESULTS_DIR, f)
    if os.path.exists(src):
        shutil.copy(src, dst)
        restored.append(f)
        print('Restored: ' + f)
    else:
        print('Not found on Drive (first run or not yet created): ' + f)

print()
if restored:
    print(str(len(restored)) + ' file(s) restored from ' + DRIVE_DIR)
else:
    print('No previous results found -- starting fresh.')

## Cell 7 — Upload Phase 1 Results

Open a file picker to manually upload your phase1_results.csv from your local machine and save it to the results directory.

In [ ]:
# Run cells in order -- this is Cell 7 of 12
import shutil, os
from google.colab import files

RESULTS_DIR = '/content/ADVC/results'
DRIVE_DIR = '/content/drive/MyDrive/research'

print('Select your phase1_results.csv when the file picker appears ...')
uploaded = files.upload()

if 'phase1_results.csv' in uploaded:
    dst = os.path.join(RESULTS_DIR, 'phase1_results.csv')
    with open(dst, 'wb') as fh:
        fh.write(uploaded['phase1_results.csv'])
    print('Saved to ' + dst)
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copy(dst, os.path.join(DRIVE_DIR, 'phase1_results.csv'))
    print('Also backed up to Drive: ' + DRIVE_DIR)
else:
    uploaded_names = list(uploaded.keys())
    if uploaded_names:
        print('WARNING: expected phase1_results.csv but got ' + str(uploaded_names))
        print('Rename your file to phase1_results.csv and re-run this cell.')
    else:
        print('No file was uploaded. Re-run this cell to try again.')

## Cell 8 — Run Phase 2a: Adversarial Training (AT) Defense

Run eval_phase2_at.py via subprocess with live output streaming. After the script completes, phase2_at_results.csv is immediately backed up to Drive.

In [ ]:
# Run cells in order -- this is Cell 8 of 12
import subprocess, sys, shutil, os

ADVC_DIR = '/content/ADVC'
DRIVE_DIR = '/content/drive/MyDrive/research'
SCRIPT = os.path.join(ADVC_DIR, 'experiments', 'eval_phase2_at.py')
RESULTS_FILE = os.path.join(ADVC_DIR, 'results', 'phase2_at_results.csv')

print('=' * 60)
print('Phase 2a: AT defense sweep')
print('=' * 60)

proc = subprocess.Popen(
    [sys.executable, SCRIPT],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=ADVC_DIR,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print()
print('[Cell 8] Process exited with return code ' + str(proc.returncode))

os.makedirs(DRIVE_DIR, exist_ok=True)
if os.path.exists(RESULTS_FILE):
    shutil.copy(RESULTS_FILE, os.path.join(DRIVE_DIR, 'phase2_at_results.csv'))
    print('[Cell 8] phase2_at_results.csv backed up to ' + DRIVE_DIR)
else:
    print('[Cell 8] WARNING: phase2_at_results.csv not found -- check run output above.')

## Cell 9 — Run Phase 2b: AT+KD Defense

Run eval_phase2_atkd.py via subprocess with live output streaming. After the script completes, phase2_atkd_results.csv is immediately backed up to Drive.

In [ ]:
# Run cells in order -- this is Cell 9 of 12
import subprocess, sys, shutil, os

ADVC_DIR = '/content/ADVC'
DRIVE_DIR = '/content/drive/MyDrive/research'
SCRIPT = os.path.join(ADVC_DIR, 'experiments', 'eval_phase2_atkd.py')
RESULTS_FILE = os.path.join(ADVC_DIR, 'results', 'phase2_atkd_results.csv')

print('=' * 60)
print('Phase 2b: AT+KD defense sweep')
print('=' * 60)

proc = subprocess.Popen(
    [sys.executable, SCRIPT],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=ADVC_DIR,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print()
print('[Cell 9] Process exited with return code ' + str(proc.returncode))

os.makedirs(DRIVE_DIR, exist_ok=True)
if os.path.exists(RESULTS_FILE):
    shutil.copy(RESULTS_FILE, os.path.join(DRIVE_DIR, 'phase2_atkd_results.csv'))
    print('[Cell 9] phase2_atkd_results.csv backed up to ' + DRIVE_DIR)
else:
    print('[Cell 9] WARNING: phase2_atkd_results.csv not found -- check run output above.')

## Cell 10 — Run Phase 3: Combined Attack

Run eval_phase3.py via subprocess with live output streaming. Requires eval_phase3.py to exist (step 9 in the CLAUDE.md build order). After the script completes, phase3_results.csv is immediately backed up to Drive.

In [ ]:
# Run cells in order -- this is Cell 10 of 12
import subprocess, sys, shutil, os

ADVC_DIR = '/content/ADVC'
DRIVE_DIR = '/content/drive/MyDrive/research'
SCRIPT = os.path.join(ADVC_DIR, 'experiments', 'eval_phase3.py')
RESULTS_FILE = os.path.join(ADVC_DIR, 'results', 'phase3_results.csv')

if not os.path.exists(SCRIPT):
    print('ERROR: ' + SCRIPT + ' does not exist yet.')
    print('Build eval_phase3.py first (step 9 in the CLAUDE.md build order) then re-run this cell.')
else:
    print('=' * 60)
    print('Phase 3: Combined attack sweep')
    print('=' * 60)

    proc = subprocess.Popen(
        [sys.executable, SCRIPT],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=ADVC_DIR,
    )

    for line in proc.stdout:
        print(line, end='', flush=True)

    proc.wait()
    print()
    print('[Cell 10] Process exited with return code ' + str(proc.returncode))

    os.makedirs(DRIVE_DIR, exist_ok=True)
    if os.path.exists(RESULTS_FILE):
        shutil.copy(RESULTS_FILE, os.path.join(DRIVE_DIR, 'phase3_results.csv'))
        print('[Cell 10] phase3_results.csv backed up to ' + DRIVE_DIR)
    else:
        print('[Cell 10] WARNING: phase3_results.csv not found -- check run output above.')

## Cell 11 — Backup All Results

Copy all four CSV result files from /content/ADVC/results to /content/drive/MyDrive/research. Run this cell any time before ending the session.

In [ ]:
# Run cells in order -- this is Cell 11 of 12
import shutil, os

ADVC_DIR = '/content/ADVC'
DRIVE_DIR = '/content/drive/MyDrive/research'

os.makedirs(DRIVE_DIR, exist_ok=True)

files = [
    'phase1_results.csv',
    'phase2_at_results.csv',
    'phase2_atkd_results.csv',
    'phase3_results.csv',
]

backed_up = []
for f in files:
    src = os.path.join(ADVC_DIR, 'results', f)
    dst = os.path.join(DRIVE_DIR, f)
    if os.path.exists(src):
        shutil.copy(src, dst)
        backed_up.append(f)
        print('Backed up: ' + f + '  ->  ' + dst)
    else:
        print('Skipped (not found): ' + f)

print()
print(str(len(backed_up)) + ' file(s) backed up to ' + DRIVE_DIR)

## Cell 12 — Preview Results

Load all four CSV files with pandas and print each as a formatted table for a quick sanity check before ending the session.

In [ ]:
# Run cells in order -- this is Cell 12 of 12
import pandas as pd
import os

RESULTS_DIR = '/content/ADVC/results'

report = {
    'Phase 1 -- No Defense': 'phase1_results.csv',
    'Phase 2a -- AT Defense': 'phase2_at_results.csv',
    'Phase 2b -- AT+KD Defense': 'phase2_atkd_results.csv',
    'Phase 3 -- Combined Attack': 'phase3_results.csv',
}

display_cols = ['compression', 'defense', 'attack', 'clean_acc', 'robust_acc', 'asr', 'robustness_gap']

for title, fname in report.items():
    path = os.path.join(RESULTS_DIR, fname)
    print()
    print('=' * 60)
    print(title)
    print('=' * 60)
    if not os.path.exists(path):
        print('  File not found: ' + path)
        continue
    df = pd.read_csv(path)
    if df.empty:
        print('  File exists but contains no data rows.')
        continue
    cols = [c for c in display_cols if c in df.columns]
    print(df[cols].to_string(index=False))
    print()
    print('  ' + str(len(df)) + ' row(s) total in ' + fname)